<a href="https://colab.research.google.com/github/Briankoskei/machine-learning-exercises/blob/main/model%20training%20air%20quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib


def generate_air_quality_data(n_samples=5000):
    np.random.seed(42)
    dates = pd.date_range(start='2024-01-01', periods=n_samples, freq='h')

    temp = 15 + 10 * np.sin(np.linspace(0, 50, n_samples)) + np.random.normal(0, 2, n_samples)
    humidity = 60 + 20 * np.cos(np.linspace(0, 50, n_samples)) + np.random.normal(0, 5, n_samples)
    wind_speed = np.abs(3 + np.random.normal(0, 1.5, n_samples))
    pm10 = 30 + 0.8 * temp + np.random.exponential(scale=15, size=n_samples)
    no2 = 25 + 0.5 * humidity + np.random.normal(0, 5, n_samples)

    pm25 = (
        5
        + 0.55 * pm10
        + 0.3 * no2
        - 0.4 * temp
        - 0.8 * wind_speed
        + np.random.normal(0, 4, n_samples)
    )

    df = pd.DataFrame({
        'temp': temp,
        'humidity': humidity,
        'wind_speed': wind_speed,
        'pm10': pm10,
        'no2': no2,
        'pm25': pm25,
    }, index=dates)

    return df


def engineer_features(df):
    df = df.copy()
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24.0)
    df['day_of_week'] = df.index.dayofweek / 6.0
    df['pm25_lag1'] = df['pm25'].shift(1)
    df['pm25_lag24'] = df['pm25'].shift(24)
    df['pm25_lag168'] = df['pm25'].shift(168)
    df = df.dropna().copy()
    return df


def train_validation_test_split(df):
    train_end = int(len(df) * 0.7)
    val_end = int(len(df) * 0.85)

    train_df = df.iloc[:train_end]
    val_df = df.iloc[train_end:val_end]
    test_df = df.iloc[val_end:]

    return train_df, val_df, test_df


def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
    }


def plot_predictions(actual, predicted, title, model_name):
    plt.figure(figsize=(12, 5))
    plt.plot(actual.index, actual.values, label='Actual PM2.5', linewidth=2)
    plt.plot(actual.index, predicted, label=f'Predicted by {model_name}', alpha=0.8)
    plt.title(title)
    plt.xlabel('Datetime')
    plt.ylabel('PM2.5 (µg/m³)')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_feature_importance(model, feature_names, model_name):
    if not hasattr(model, 'feature_importances_'):
        return

    importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    importances.head(10).plot(kind='bar', color='steelblue')
    plt.title(f'Top 10 Feature Importances for {model_name}')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()


def main():
    df = engineer_features(generate_air_quality_data())

    features = [
        'temp', 'humidity', 'wind_speed', 'pm10', 'no2',
        'hour_sin', 'hour_cos', 'day_of_week', 'pm25_lag1', 'pm25_lag24', 'pm25_lag168'
    ]

    X = df[features]
    y = df['pm25']

    train_df, val_df, test_df = train_validation_test_split(df)
    X_train, X_val, X_test = train_df[features], val_df[features], test_df[features]
    y_train, y_val, y_test = train_df['pm25'], val_df['pm25'], test_df['pm25']

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    candidate_models = [
        ('Baseline (Persistence - Lag1)', None),
        ('Linear Regression', LinearRegression()),
        ('Ridge Regression', Ridge(alpha=10.0)),
        ('Decision Tree', DecisionTreeRegressor(max_depth=10, min_samples_leaf=2, random_state=42)),
        ('Random Forest', RandomForestRegressor(n_estimators=300, max_depth=18, min_samples_leaf=2, random_state=42)),
        ('Gradient Boosting', GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42,
        ))
    ]

    fitted_models = {}
    results = []

    for name, model in candidate_models:
        if name == 'Baseline (Persistence - Lag1)':
            val_pred = X_val['pm25_lag1'].fillna(y_val.mean()).to_numpy()
            test_pred = X_test['pm25_lag1'].fillna(y_test.mean()).to_numpy()
        else:
            if 'Linear' in name or 'Ridge' in name:
                model.fit(X_train_scaled, y_train)
                val_pred = model.predict(X_val_scaled)
                test_pred = model.predict(X_test_scaled)
            else:
                model.fit(X_train, y_train)
                val_pred = model.predict(X_val)
                test_pred = model.predict(X_test)

            val_pred = np.clip(val_pred, 0, None)
            test_pred = np.clip(test_pred, 0, None)
            fitted_models[name] = model

        val_metrics = calculate_metrics(y_val, val_pred)
        test_metrics = calculate_metrics(y_test, test_pred)

        results.append({
            'Model': name,
            'Validation RMSE': round(val_metrics['RMSE'], 3),
            'Validation MAE': round(val_metrics['MAE'], 3),
            'Validation R²': round(val_metrics['R2'], 4),
            'Test RMSE': round(test_metrics['RMSE'], 3),
            'Test MAE': round(test_metrics['MAE'], 3),
            'Test R²': round(test_metrics['R2'], 4),
        })

    results_df = pd.DataFrame(results).sort_values('Validation RMSE').reset_index(drop=True)
    print(results_df.to_string(index=False))

    best_model_name = results_df.iloc[0]['Model']
    best_model = fitted_models.get(best_model_name)

    if best_model is None:
        best_pred = X_test['pm25_lag1'].fillna(y_test.mean()).to_numpy()
    elif best_model_name in ['Linear Regression', 'Ridge Regression']:
        best_pred = best_model.predict(X_test_scaled)
    else:
        best_pred = best_model.predict(X_test)

    best_pred = np.clip(best_pred, 0, None)
    plot_predictions(y_test, best_pred, f'PM2.5 Forecast vs Actual on Test Set ({best_model_name})', best_model_name)
    if best_model is not None and hasattr(best_model, 'feature_importances_'):
        plot_feature_importance(best_model, features, best_model_name)

    # Resolve the NameError: '__file__' is not defined in notebooks
    try:
        base_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        base_dir = os.getcwd()

    output_dir = os.path.join(base_dir, 'saved_model')
    os.makedirs(output_dir, exist_ok=True)

    joblib.dump(best_model if best_model is not None else 'baseline', os.path.join(output_dir, 'best_model.pkl'))
    joblib.dump(scaler, os.path.join(output_dir, 'scaler.pkl'))
    joblib.dump(features, os.path.join(output_dir, 'features.pkl'))
    results_df.to_csv(os.path.join(output_dir, 'model_results.csv'), index=False)

    print(f"\nSaved model and artifacts to: {output_dir}")


if __name__ == '__main__':
    main()

                        Model  Validation RMSE  Validation MAE  Validation R²  Test RMSE  Test MAE  Test R²
             Ridge Regression            4.061           3.247         0.8143      4.034     3.198   0.8360
            Linear Regression            4.064           3.249         0.8140      4.034     3.199   0.8361
            Gradient Boosting            4.146           3.301         0.8064      4.130     3.254   0.8282
                Random Forest            4.278           3.385         0.7939      4.192     3.354   0.8230
                Decision Tree            5.435           4.299         0.6673      5.230     4.198   0.7244
Baseline (Persistence - Lag1)           12.505           9.517        -0.7612     13.851    10.343  -0.9331

Saved model and artifacts to: /content/saved_model


In [ ]:
from sklearn.model_selection import GridSearchCV

# 1. Regenerate and prepare the data matching the main pipeline
df = engineer_features(generate_air_quality_data())
features = [
    'temp', 'humidity', 'wind_speed', 'pm10', 'no2',
    'hour_sin', 'hour_cos', 'day_of_week', 'pm25_lag1', 'pm25_lag24', 'pm25_lag168'
]

train_df, val_df, test_df = train_validation_test_split(df)
X_train, X_val, X_test = train_df[features], val_df[features], test_df[features]
y_train, y_val, y_test = train_df['pm25'], val_df['pm25'], test_df['pm25']

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 2. Define the Ridge hyperparameter search grid
ridge = Ridge()
param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1.0, 5.0, 10.0, 20.0, 50.0, 100.0, 200.0]
}

# 3. Perform Grid Search with Cross Validation
grid_search = GridSearchCV(
    estimator=ridge,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

# 4. Extract and print best parameters
best_alpha = grid_search.best_params_['alpha']
best_ridge_model = grid_search.best_estimator_

print(f"Best alpha found: {best_alpha}")

# 5. Evaluate on Validation and Test set
val_pred = np.clip(best_ridge_model.predict(X_val_scaled), 0, None)
test_pred = np.clip(best_ridge_model.predict(X_test_scaled), 0, None)

val_metrics = calculate_metrics(y_val, val_pred)
test_metrics = calculate_metrics(y_test, test_pred)

print("\nTuned Ridge Model Performance:")
print(f"Validation RMSE: {val_metrics['RMSE']:.4f} | Validation R²: {val_metrics['R2']:.4f}")
print(f"Test RMSE:       {test_metrics['RMSE']:.4f} | Test R²:       {test_metrics['R2']:.4f}")

Best alpha found: 5.0

Tuned Ridge Model Performance:
Validation RMSE: 4.0624 | Validation R²: 0.8141
Test RMSE:       4.0338 | Test R²:       0.8361


In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. Re-evaluate models to reconstruct 'results'
candidate_models = [
    ('Baseline (Persistence - Lag1)', None),
    ('Linear Regression', LinearRegression()),
    ('Ridge Regression (Baseline)', Ridge(alpha=10.0)),
    ('Ridge Regression (Tuned)', Ridge(alpha=5.0)),
    ('Decision Tree', DecisionTreeRegressor(max_depth=10, min_samples_leaf=2, random_state=42)),
    ('Random Forest', RandomForestRegressor(n_estimators=300, max_depth=18, min_samples_leaf=2, random_state=42)),
    ('Gradient Boosting', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    ))
]

results = []

for name, model in candidate_models:
    if name == 'Baseline (Persistence - Lag1)':
        v_pred = X_val['pm25_lag1'].fillna(y_val.mean()).to_numpy()
        t_pred = X_test['pm25_lag1'].fillna(y_test.mean()).to_numpy()
    else:
        if 'Linear' in name or 'Ridge' in name:
            model.fit(X_train_scaled, y_train)
            v_pred = model.predict(X_val_scaled)
            t_pred = model.predict(X_test_scaled)
        else:
            model.fit(X_train, y_train)
            v_pred = model.predict(X_val)
            t_pred = model.predict(X_test)

        v_pred = np.clip(v_pred, 0, None)
        t_pred = np.clip(t_pred, 0, None)

    v_metrics = calculate_metrics(y_val, v_pred)
    t_metrics = calculate_metrics(y_test, t_pred)

    results.append({
        'Model': name,
        'Validation RMSE': round(v_metrics['RMSE'], 3),
        'Validation MAE': round(v_metrics['MAE'], 3),
        'Validation R²': round(v_metrics['R2'], 4),
        'Test RMSE': round(t_metrics['RMSE'], 3),
        'Test MAE': round(t_metrics['MAE'], 3),
        'Test R²': round(t_metrics['R2'], 4),
    })

# 2. Convert to DataFrame and print comparison table
results_df = pd.DataFrame(results).sort_values('Validation RMSE').reset_index(drop=True)
print(results_df.to_string(index=False))

# 3. EXPORT RESULTS TO CSV
csv_filename = "model_evaluation_results.csv"
results_df.to_csv(csv_filename, index=False)
print(f"\nSuccessfully saved evaluation results including Tuned Ridge to: {csv_filename}")

                        Model  Validation RMSE  Validation MAE  Validation R²  Test RMSE  Test MAE  Test R²
  Ridge Regression (Baseline)            4.061           3.247         0.8143      4.034     3.198   0.8360
     Ridge Regression (Tuned)            4.062           3.248         0.8141      4.034     3.198   0.8361
            Linear Regression            4.064           3.249         0.8140      4.034     3.199   0.8361
            Gradient Boosting            4.146           3.301         0.8064      4.130     3.254   0.8282
                Random Forest            4.278           3.385         0.7939      4.192     3.354   0.8230
                Decision Tree            5.435           4.299         0.6673      5.230     4.198   0.7244
Baseline (Persistence - Lag1)           12.505           9.517        -0.7612     13.851    10.343  -0.9331

Successfully saved evaluation results including Tuned Ridge to: model_evaluation_results.csv
